# Clustering of TOP 5 European Leagues Players
**Authors:** Cezary Kuźmowicz, Filip Żebrowski, Dariusz Doktorski

## “The ball doesn’t go in by chance” ~ Johan Cruyff

Football is the most popular sport on Earth. Millions of people around the globe play it on daily basis. Most of countries have their own national leagues. But the best of the best take place in Europe. In football's nomenclature, while sharing some graphs and statistics, often used concept is "TOP 5 European Leagues". That means clearly five best national competitions in Europe. 

This group consist of English Premier League, Spanish LaLiga, Italian Serie A, German Bundesliga and French Ligue 1. In this report we will conduct clustering analysis using players statistics from season 2021/22.

In order to achieve satisfactory results many tests and visualizations will be presented. As the clustering method we've chosen k-means, mainly due to medium size of dataset and simply interpretation.

[dataset used for analysis](https://www.kaggle.com/datasets/vivovinco/20212022-football-player-stats)

### Installing necessary packages

In the beginning we have to install and access needed packages for whole analysis:

In [1]:
import random 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm

from numpy.random import uniform
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

# Set default plotting style
sns.set_theme(style="whitegrid")

### Setting up parameters
By centralizing all our configuration constants, we eliminate "magic numbers" from our code and make it fully reproducible and easy to tweak in one place.

In [2]:
# File settings
FILE_PATH = "data/2021-2022 Football Player Stats.csv"
CSV_SEP = ";"
CSV_DECIMAL = "."
CSV_ENCODING = "latin1"

# Data cleaning & Model parameters
MIN_MINUTES_PLAYED = 180
HOPKINS_SAMPLE_RATIO = 0.1
HOPKINS_N_NEIGHBORS = 2
KMEANS_N_INIT = 10
MAX_CLUSTERS = 10
OPTIMAL_K = 3
RANDOM_SEED = 2137

# Feature selection
SELECTED_FEATURES = ['Shots', 'PasTotAtt', 'Assists', 'Tkl', 'Blocks', 'Fls', 'Off', 'AerWon%']
VISUALIZATION_FEATURES = ["Min", "90s", "Goals", "PasTotAtt", "AerWon%"]

# Set Seeds for reproducibility 
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# CLUSTERING PREPARATIONS

## Data Preparation

### Loading dataset into Python

The dataset comes from Kaggle. It was prepared based on data from [fbref](https://fbref.com/en/) - online website gathering huge amount of informations from plenty of sports.

In [3]:
# Load the dataset using predefined parameters to handle special characters and formats
raw_stats = pd.read_csv(FILE_PATH, sep=CSV_SEP, decimal=CSV_DECIMAL, encoding=CSV_ENCODING)

The raw dataset includes nearly 3000 observations (individual players), each described by 143 variables. It's important to add that every statistic is calculated "per 90 minutes". This means that the dataset's author already unified the data.

The first step during our data preparation will be excluding footballers who played less than a specific threshold of minutes through the whole season. That operation will assure us that we don't analyze a player who, for example, scored 4 goals in his only played match.

In [4]:
# Filter players based on the configured minimum minutes played
stats_filtered = raw_stats[raw_stats['Min'] >= MIN_MINUTES_PLAYED].copy()

By conducting such an operation we got rid of nearly 600 observations. That will definitely improve overview of our dataset and quality of analysis.

Before removing some of them, let's dive into overview of the data

In [5]:
display(stats_filtered.head())
display(stats_filtered.describe())
print(stats_filtered.info())

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Off,Crs,TklW,PKwon,PKcon,OG,Recov,AerWon,AerLost,AerWon%
0,1,Max Aarons,ENG,DF,Norwich City,Premier League,22.0,2000,34,32,...,0.03,1.41,1.16,0.0,0.06,0.03,5.53,0.47,1.59,22.7
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,34.0,1987,34,34,...,0.00,0.06,1.39,0.0,0.03,0.00,6.77,2.02,1.36,59.8
2,3,Salis Abdul Samed,GHA,MF,Clermont Foot,Ligue 1,22.0,2000,31,29,...,0.00,0.36,1.24,0.0,0.00,0.00,8.76,0.88,0.88,50.0
3,4,Laurent Abergel,FRA,MF,Lorient,Ligue 1,29.0,1993,34,34,...,0.03,0.79,2.23,0.0,0.00,0.00,8.87,0.43,0.43,50.0
5,6,Dickson Abiama,NGA,FW,Greuther Fürth,Bundesliga,23.0,1998,24,5,...,1.85,0.25,0.86,0.0,0.00,0.00,4.81,2.72,4.94,35.5


,Rk,Age,Born,MP,Starts,Min,90s,Goals,Shots,SoT,...,Off,Crs,TklW,PKwon,PKcon,OG,Recov,AerWon,AerLost,AerWon%
count,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000,...,2350.000000,2350.00000,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000,2350.000000
mean,1459.128936,26.764255,1994.732340,22.680426,17.000426,1520.581277,16.894809,0.121417,1.178719,0.382000,...,0.181017,1.10397,1.019813,0.010970,0.014298,0.004289,7.516119,1.611179,1.701579,43.475362
std,839.305104,4.324699,4.319047,9.456337,10.354200,877.546957,9.749572,0.172684,0.974991,0.397837,...,0.291827,1.13350,0.610564,0.038134,0.038873,0.022560,2.504638,1.329008,1.316661,19.799631
min,1.000000,17.000000,1981.000000,2.000000,0.000000,180.000000,2.000000,0.000000,0.000000,0.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.430000,0.000000,0.000000,0.000000
25%,739.250000,24.000000,1992.000000,15.000000,8.000000,764.250000,8.500000,0.000000,0.420000,0.070000,...,0.000000,0.19000,0.600000,0.000000,0.000000,0.000000,5.530000,0.660000,0.950000,33.300000
50%,1458.500000,26.000000,1995.000000,24.000000,16.000000,1447.500000,16.100000,0.050000,0.890000,0.240000,...,0.050000,0.76000,0.975000,0.000000,0.000000,0.000000,7.500000,1.310000,1.410000,45.550000
75%,2170.750000,30.000000,1998.000000,31.000000,26.000000,2221.250000,24.700000,0.180000,1.830000,0.597500,...,0.240000,1.79000,1.400000,0.000000,0.000000,0.000000,9.260000,2.270000,2.087500,57.900000
max,2921.000000,40.000000,2004.000000,38.000000,38.000000,3420.000000,38.000000,1.430000,5.100000,2.350000,...,2.070000,9.13000,3.390000,0.500000,0.500000,0.500000,15.700000,12.400000,9.720000,100.000000


<class 'pandas.core.frame.DataFrame'>
Int64Index: 2350 entries, 0 to 2920
Columns: 143 entries, Rk to AerWon%
dtypes: float64(133), int64(5), object(5)
memory usage: 2.6+ MB
None


As you can see, there is too much information. Analyzing dataset with such a number of variables is very impractical. But there are some overall insights are can obtain:
   
* many variables are right-skewed
* in almost every statistic max value is hard outlier. It isn't error; that's just world-class players

In conducting clustering important part is assuring that none NAs occur

In [6]:
# Print variables with missing values (if any)
missing_vals = stats_filtered.isna().sum()[stats_filtered.isna().sum() > 0]
print(missing_vals)

# Ensure no missing values exist before proceeding to clustering
assert missing_vals.sum() == 0, "Pipeline Error: Raw data contains missing values!"
print("Data validation passed: No missing values found.")

Series([], dtype: int64)
Data validation passed: No missing values found.


As we can see - there are no NA values. Probably the author from Kaggle already handled it.

## Handling number of variables

In our dataset we have nearly 150 columns. That is pretty impressive amount but plenty of them are too detailed/provide no valuable information. In order to choose the best we studied every columns. What's more, we're football fans for more than 12 years. Thanks to that experience we were able to choose the most valuable variables. 

After plenty of testing (including calculating Hopkins statistics for each set of columns), we decided to limit number of variables from 143 to 8. Part of Data Scientist's job is using previous experience and knowledge to better understand and solve problems. So did we here!

We've tried to choose as diverse and valuable variables as we could. Let's decode them:  
* **Shots**: shots taken by player, per 90 min  
* **PasTotAtt**: total number of pass attempts, per 90 min   
* **Assists**: assists given by player, per 90 min   
* **Tkl**: number of times a ball has been picked up (tackled), per 90 min   
* **Block**: number of times player has blocked a pass/shot, per 90 min    
* **Fls**: number of fauls comitted by player, per 90 min    
* **Off**: number of times player has been caught on offside position, per 90 min    
* **AerWon%**: share of won aerial head duels, in %

By creating new data frame with less variables, we could easily continue the analysis.

In [7]:
reduced_stats = stats_filtered[SELECTED_FEATURES].copy()
reduced_stats.head()

,Shots,PasTotAtt,Assists,Tkl,Blocks,Fls,Off,AerWon%
0,0.41,45.0,0.06,2.16,2.69,0.97,0.03,22.7
1,0.54,47.0,0.00,1.87,1.87,1.30,0.00,59.8
2,0.66,61.0,0.00,2.01,0.99,1.64,0.00,50.0
3,0.91,49.8,0.06,3.57,1.68,1.40,0.03,50.0
5,2.22,17.2,0.12,1.73,1.23,2.22,1.85,35.5


### Hopkins statistics
In order to check how clusterable is the dataset, we'll compute Hopkins test. It compares the distances between randomly sampled points in the dataset and synthetic points generated uniformly in the feature space. A value close to **1** indicates strong clustering tendency, while a value near **0.5** suggests the data is uniformly distributed and unlikely to form meaningful clusters.

In [8]:
def hopkins_statistic(X: pd.DataFrame, sample_ratio: float = 0.1, n_neighbors: int = 2) -> float:
    """
    Computes the Hopkins statistic to measure the cluster tendency of a dataset using vectorized operations.
    
    Parameters:
    X (pd.DataFrame): The input dataset to evaluate.
    sample_ratio (float): The fraction of the dataset to sample (default 0.1).
    n_neighbors (int): The number of nearest neighbors to search for (default 2).
    
    Returns:
    float: The Hopkins statistic (H). A value near 1 indicates strong clustering tendency, 
           while a value near 0.5 indicates uniform data.
    """
    d = X.shape[1]
    n = len(X)
    m = int(sample_ratio * n)
    
    # Fit NearestNeighbors
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(X.values)
    
    # Vectorized Generation of 'm' simulated uniform points
    min_vals = np.amin(X.values, axis=0)
    max_vals = np.amax(X.values, axis=0)
    simulated_points = uniform(min_vals, max_vals, (m, d))
    
    # Vectorized Selection of 'm' real points
    rand_X_indices = random.sample(range(n), m)
    real_points = X.iloc[rand_X_indices].values
    
    # Vectorized Distance Calculation for all points simultaneously
    u_dist, _ = nbrs.kneighbors(simulated_points, n_neighbors=n_neighbors)
    w_dist, _ = nbrs.kneighbors(real_points, n_neighbors=n_neighbors)
    
    # Extract the distance to the nth neighbor (index n_neighbors - 1)
    # and sum all distances without using any slow sequential Python loops
    sum_ujd = np.sum(u_dist[:, n_neighbors - 1])
    sum_wjd = np.sum(w_dist[:, n_neighbors - 1])
    
    H = sum_ujd / (sum_ujd + sum_wjd)
    return float(H)

# Execute function using dynamic parameters
hopkins_val = hopkins_statistic(reduced_stats, sample_ratio=HOPKINS_SAMPLE_RATIO, n_neighbors=HOPKINS_N_NEIGHBORS)
print(f"Hopkins Statistic: {hopkins_val:.4f}")

Hopkins Statistic: 0.8134


The value of Hopkins statistic is high. That's good predictor for our clustering.

## Data Visualisation

In order present some of the statistics of charts, we'll create data frame variable for those purposes.

In [9]:
# Slice for visualization
player_stats = stats_filtered[VISUALIZATION_FEATURES].copy()

# Feature engineering for visualization
player_stats['total_goals'] = player_stats['Goals'] * player_stats['90s']

### Visualizing Player Statistics
To understand our dataset better and ensure results aren't biased by outliers, we will visualize a few key statistics:

* **Minutes played**: It's important to see if our results won't be biased by footballers who played very limited time.
* **Goals**: We multiply "goals/90min" by "played 90s" to present the most goalscoring players.
* **Passes**: How many passes players make per 90 minutes.
* **Aerial Duels Won (%)**: A key attribute for center-backs and target-man forwards. 

In [ ]:
def plot_distribution(data: pd.DataFrame, column: str, title: str, xlabel: str, color: str, binwidth: int = None) -> None:
    """
    Plots the distribution of a given column in a dataframe to avoid repetitive code.
    """
    plt.figure(figsize=(8, 5))
    sns.histplot(data[column], binwidth=binwidth, color=color, edgecolor="black")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Number of players")
    plt.show()

# Use a loop over a configuration list to dynamically plot multiple features
plot_configs = [
    {
        'column': 'Min', 
        'title': 'Distribution of Minutes Played (90 minutes intervals)', 
        'xlabel': 'Minutes Played (90 minutes bins)', 
        'color': 'skyblue', 
        'binwidth': 90
    },
    {
        'column': 'total_goals', 
        'title': 'Distribution of Goals', 
        'xlabel': 'Goals', 
        'color': 'red', 
        'binwidth': 1
    },
    {
        'column': 'PasTotAtt', 
        'title': 'Distribution of passes per 90 minutes', 
        'xlabel': 'Passes', 
        'color': 'green', 
        'binwidth': 5
    },
    {
        'column': 'AerWon%', 
        'title': 'Distribution of Aerial Duels Won (%)', 
        'xlabel': 'Aerial Duels Won (%)', 
        'color': 'orange', 
        'binwidth': 5
    }
]

for config in plot_configs:
    plot_distribution(
        data=player_stats,
        column=config['column'],
        title=config['title'],
        xlabel=config['xlabel'],
        color=config['color'],
        binwidth=config['binwidth']
    )

### Interpretation of the Distributions

* **Minutes Played**: The played time is more or less equally distributed across the spectrum. The biggest differences and spikes are visible at the very beginning (players with very few minutes) and the end (undisputed starters who play almost every match), which is highly intuitive.
* **Goals (Total)**: As expected, the vast majority of players (defenders, goalkeepers, defensive midfielders) didn't score any goals or scored very few. The plot features an extremely long right tail—these aren't errors in the dataset, but rather the super-star forwards who score 20+ goals a season.
* **Passes (per 90 min)**: This statistic is much more normally distributed than goals. Passing is a core mechanic for almost all players on the pitch, not just a specific trait of midfielders. We can observe a slight right skew, which represents deep-lying playmakers and center-backs who circulate the ball the most frequently.
* **Aerial Duels Won (%)**: This displays a somewhat normal distribution centered around the 40-60% mark. However, there are notable and massive spikes at the absolute extreme 0%. This is mostly due to players who attempted very few aerial duels throughout the entire season, which heavily distorted their personal win percentages.

### Correlation Matrix
Before moving to clustering, let's verify if there is strong collinearity among the chosen features.

In [ ]:
def plot_correlation_matrix(data: pd.DataFrame, title: str) -> None:
    """
    Plots a heatmap for the correlation matrix of the provided dataframe.
    """
    plt.figure(figsize=(10, 8))
    corr_matrix = data.corr()
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title(title)
    plt.show()

plot_correlation_matrix(reduced_stats, "Correlation Matrix of Selected Features")

### Interpretation of the Correlation Matrix

Analyzing the correlation matrix gives us confidence that our selected variables are well-suited for clustering:

* **Logical Groupings**: We can clearly see moderate to strong positive correlations between offensive metrics (e.g., `Shots`, `Assists`, and `Off`), as well as between defensive metrics (e.g., `Tkl` and `Blocks`). This makes sense, as players usually specialize in either attack or defense.
* **Negative Correlations**: There are noticeable negative correlations between offensive and defensive stats (e.g., `Shots` vs. `Blocks`). This divergence is exactly what K-Means needs to effectively separate attackers from defenders.
* **No Extreme Multicollinearity**: Crucially, there are no correlations extremely close to 1.0 or -1.0. If two variables were perfectly correlated, one of them would be redundant and could artificially over-weight a specific trait during the K-Means distance calculations. The current balance is ideal for creating distinct, meaningful clusters.

# CLUSTERING ANALYSIS

## Pre-clustering details
### Scaling the data

Scaling data is necessary - variable such as "AerWon%" would have stronger impact on results of analysis due to it's format (range from 0 to 100). Rest variables are calculated per 90 minutes.

In [12]:
# Scale the data using our clean dataset
scaler = StandardScaler()
stats_scaled_array = scaler.fit_transform(reduced_stats)
stats_scaled = pd.DataFrame(stats_scaled_array, columns=SELECTED_FEATURES)

display(stats_scaled.head())
display(stats_scaled.describe().round(2))

,Shots,PasTotAtt,Assists,Tkl,Blocks,Fls,Off,AerWon%
0,-0.788605,0.116330,-0.226095,0.513603,1.774285,-0.486416,-0.517598,-1.049504
1,-0.655242,0.246630,-0.727659,0.207376,0.641100,-0.050253,-0.620420,0.824668
2,-0.532138,1.158728,-0.727659,0.355209,-0.575002,0.399127,-0.620420,0.329603
3,-0.275671,0.429049,-0.226095,2.002498,0.378532,0.081918,-0.517598,0.329603
4,1.068217,-1.694836,0.275469,0.059542,-0.243338,1.165716,5.720292,-0.402889


,Shots,PasTotAtt,Assists,Tkl,Blocks,Fls,Off,AerWon%
count,2350.00,2350.00,2350.00,2350.00,2350.00,2350.00,2350.00,2350.00
mean,0.00,0.00,0.00,0.00,-0.00,0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.21,-2.08,-0.73,-1.77,-1.94,-1.77,-0.62,-2.20
25%,-0.78,-0.72,-0.73,-0.68,-0.65,-0.66,-0.62,-0.51
50%,-0.30,-0.08,-0.31,-0.02,0.03,-0.06,-0.45,0.10
75%,0.67,0.62,0.44,0.67,0.67,0.58,0.20,0.73
max,4.02,3.74,10.14,4.44,5.27,4.55,6.47,2.86


After scaling we could see that our data is more reliable showing variance in variables. It'll be perfect for clustering.

### Optimal number of clusters

In order to get optimal number of clusters, we will use the Elbow Method (Variance Explained) and Silhouette Scores. Decision about number of clusters is not easy, but checking multiple metrics provides a clearer picture.

In [ ]:
def evaluate_clusters(data: pd.DataFrame, max_k: int, random_seed: int, n_init: int) -> None:
    """
    Evaluates optimal number of clusters using WCSS and Silhouette Score to avoid hardcoding test boundaries.
    """
    wcss = []
    silhouette_scores = []
    K = range(2, max_k + 1)
    
    for k in K:
        kmeans = KMeans(n_clusters=k, random_state=random_seed, n_init=n_init)
        kmeans.fit(data)
        wcss.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(data, kmeans.labels_))
        
    # Plotting metrics
    _, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].plot(K, wcss, 'bo-', markersize=8)
    axes[0].set_title('Elbow Method')
    axes[0].set_xlabel('Number of clusters')
    axes[0].set_ylabel('WCSS')
    
    axes[1].plot(K, silhouette_scores, 'ro-', markersize=8)
    axes[1].set_title('Silhouette Score')
    axes[1].set_xlabel('Number of clusters')
    axes[1].set_ylabel('Score')
    
    plt.tight_layout()
    plt.show()

# Call the function utilizing the configured parameters
evaluate_clusters(stats_scaled, max_k=MAX_CLUSTERS, random_seed=RANDOM_SEED, n_init=KMEANS_N_INIT)

Based on both the Elbow Method and Silhouette Scores, 3 clusters seem to be the most optimal choice. The WCSS curve starts flattening out after k=3, and the Silhouette Score is also highest for k=3.

## K-Means Clustering

In [14]:
# Apply configured parameters
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_SEED, n_init=KMEANS_N_INIT)

# Predict and assign clusters
reduced_stats['Cluster'] = kmeans_final.fit_predict(stats_scaled)

# Convert to string to ensure visualizations treat clusters as discrete categories, not a continuous gradient
reduced_stats['Cluster_Category'] = reduced_stats['Cluster'].astype(str)

### Silhouette Profile
To check how well an observation fits into its assigned cluster, we will calculate and visualize the Silhouette width. It has a range from -1 to 1, and we can interpret certain values in this way:   

* **s(i) ~ +1**: observation is much closer to points in its own cluster than to points in other clusters (good fit, perfect situation).   
* **s(i) ~ 0**: observation is on the decision boundary between two neighboring clusters.   
* **s(i) ~ -1**: observation is assigned to the wrong cluster.   

Knowing the theory behind Silhouette width, we can now calculate and visualize it.

In [ ]:
def plot_silhouette_profile(data: pd.DataFrame, labels: np.ndarray, n_clusters: int) -> None:
    """
    Generates a silhouette plot for the distinct clusters to evaluate clustering quality.
    """
    fig, ax1 = plt.subplots(1, 1, figsize=(8, 6))
    y_lower = 10

    sample_silhouette_values = silhouette_samples(data, labels)
    avg_silhouette = silhouette_score(data, labels)

    for i in range(n_clusters):
        ith_cluster_silhouette_values = sample_silhouette_values[labels == i]
        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_values,
                          facecolor=color, edgecolor=color, alpha=0.7)
        
        # Label the silhouette plots with their cluster numbers at the middle
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10

    ax1.set_title(f"Silhouette Plot for {n_clusters} Clusters")
    ax1.set_xlabel("The silhouette coefficient values")
    ax1.set_ylabel("Cluster label")
    
    # The vertical line for average silhouette score of all the values
    ax1.axvline(x=avg_silhouette, color="red", linestyle="--")
    plt.show()

    print(f"Average Silhouette Width: {avg_silhouette:.2f}")

# Execute using our generated labels
plot_silhouette_profile(stats_scaled, kmeans_final.labels_, OPTIMAL_K)

Average Silhouette Width: 0.34

**Silhouette Plot Interpretation:**
The dashed red line represents the average silhouette score for the entire dataset. We can observe that all three clusters have a majority of observations passing this average line, which is a sign of good structural integrity. 

* **Cluster 2** (the thickest shape at the top) has the largest number of observations. It maintains a solid positive shape, though its maximum silhouette values are slightly lower than the other clusters.
* **Cluster 1** (the thinnest shape in the middle) has the fewest observations but shows the highest cohesion. Its values stretch the furthest to the right (reaching nearly 0.45), indicating these data points are extremely distinct and well-matched to their cluster.
* **Cluster 0** (the medium-thick shape at the bottom) shows a healthy structure with many points stretching well past the average line. There are very few negative values across the board, meaning almost no players were outright assigned to the wrong cluster.